In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

batch_size = 16
IMG_SIZE = (224, 224)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    r"G:\My Drive\Bone_fracture\train",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    r"G:\My Drive\Bone_fracture\val",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

Found 8863 images belonging to 2 classes.
Found 600 images belonging to 2 classes.


In [3]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [4]:
base_model.trainable = False

In [5]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)

x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)

x = Dropout(0.3)(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "model/bone_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [16]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=[early_stopping, checkpoint]
)

Epoch 1/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 0s 997ms/step - accuracy: 0.6154 - loss: 0.6614
Epoch 1: val_loss improved from 0.69040 to 0.60192, saving model to best_bone_fracture_model.keras

Epoch 1: finished saving model to best_bone_fracture_model.keras
554/554 ━━━━━━━━━━━━━━━━━━━━ 585s 1s/step - accuracy: 0.6154 - loss: 0.6614 - val_accuracy: 0.6550 - val_loss: 0.6019
Epoch 2/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7163 - loss: 0.5623
Epoch 2: val_loss improved from 0.60192 to 0.57823, saving model to best_bone_fracture_model.keras

Epoch 2: finished saving model to best_bone_fracture_model.keras
554/554 ━━━━━━━━━━━━━━━━━━━━ 591s 1s/step - accuracy: 0.7163 - loss: 0.5623 - val_accuracy: 0.6867 - val_loss: 0.5782
Epoch 3/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7829 - loss: 0.4689
Epoch 3: val_loss did not improve from 0.57823
554/554 ━━━━━━━━━━━━━━━━━━━━ 622s 1s/step - accuracy: 0.7829 - loss: 0.4689 - val_accuracy: 0.6817 - val_loss: 0.6132
Epoch 4/20


In [17]:
base_model.trainable = True

In [18]:
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [19]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img(
    r"G:\My Drive\Bone_fracture\broken-bones-s2-fractures.webp",
    target_size=(224,224)
)

img = image.img_to_array(img)

img = img/255.0

img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

if prediction[0][0] > 0.5:
    print("Not Fractured")
else:
    print("Fractured")

print("Probability:", prediction[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Fractured
Probability: 0.48929867
